# 05. Table Recognition (Model A - Step 4)

`02_layout_detection.ipynb`에서 감지한 `table` 영역에서 행/열 구조를 추출합니다.

**방법:** OpenCV 선 감지 + Tesseract 셀 OCR (추가 설치 없음)  
**입력:** `outputs/formula_ocr/batch_formula.json`  
**출력:** `table` 요소의 `text` 필드에 Markdown 표 형식으로 저장

In [ ]:
import fitz
import io
import os
import json
import re
import cv2
import numpy as np
import pytesseract
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

PDF_PATH    = "../data/sample/2015개정경제수학-광주교육청.pdf"
FORMULA_DIR = "../outputs/formula_ocr"
LAYOUT_DIR  = "../outputs/layout"
OUTPUT_DIR  = "../outputs/table_recognition"
os.makedirs(OUTPUT_DIR, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"총 페이지 수: {len(doc)}")

In [ ]:
def page_to_image(doc, page_num, dpi=300):
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def crop_element(img, bbox_px, pad=8):
    x1, y1, x2, y2 = bbox_px
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(img.width, x2 + pad), min(img.height, y2 + pad)
    return img.crop((x1, y1, x2, y2))

## 0. 표 있는 페이지 탐색

기존 배치(10~14페이지)에 표가 없으므로, 더 넓은 범위에서 레이아웃 감지를 빠르게 실행합니다.

In [ ]:
def find_table_regions(img, min_intersections=6):
    """OpenCV로 표 감지 — 수평+수직선 교점이 충분하면 표로 판단, bbox 반환"""
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h, w = thresh.shape
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(w // 15, 20), 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(h // 15, 20)))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)

    intersections = cv2.bitwise_and(h_lines, v_lines)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 30))
    merged = cv2.dilate(intersections, kernel, iterations=3)

    contours, _ = cv2.findContours(merged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    regions = []
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        # 교점 수 확인
        roi = intersections[y:y+ch, x:x+cw]
        pts, _ = cv2.findContours(roi, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        if len(pts) >= min_intersections:
            regions.append([x, y, x+cw, y+ch])
    return regions


SCAN_RANGE = range(14, 60)  # 15~60페이지 스캔
table_pages = []  # [(page_num, [bbox_px, ...])]

for page_num in SCAN_RANGE:
    img = page_to_image(doc, page_num, dpi=100)  # 탐색용 저해상도
    regions = find_table_regions(img)
    if regions:
        table_pages.append((page_num, regions))
        print(f"  [{page_num+1}페이지] 표 {len(regions)}개 감지")

print(f"\n표 있는 페이지: {[p+1 for p, _ in table_pages]}")

In [ ]:
if not table_pages:
    print("표를 찾지 못했습니다. SCAN_RANGE를 늘려보세요.")
else:
    TEST_PAGE, TEST_REGIONS = table_pages[0]
    print(f"TEST_PAGE = {TEST_PAGE} ({TEST_PAGE+1}페이지), 표 영역 {len(TEST_REGIONS)}개")

## 1. 표 구조 추출 함수

**흐름:** 이진화 → 수평/수직선 감지 → 셀 bbox 추출 → 셀별 Tesseract OCR → Markdown 조립

In [ ]:
TESS_CONFIG = r'--oem 3 --psm 7'

def detect_cells(crop_img):
    gray = cv2.cvtColor(np.array(crop_img), cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h, w = thresh.shape
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(w // 20, 10), 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(h // 20, 10)))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)

    grid = cv2.add(h_lines, v_lines)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    dilated = cv2.dilate(grid, kernel, iterations=2)
    cells_mask = cv2.bitwise_not(dilated)

    contours, _ = cv2.findContours(cells_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    cells = []
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch
        if area < 500 or area > w * h * 0.9:
            continue
        cells.append((x, y, cw, ch))
    return cells


def cells_to_grid(cells, img_h):
    if not cells:
        return []
    tol = max(img_h * 0.02, 5)

    row_centers = sorted(set(y + h // 2 for x, y, w, h in cells))
    row_groups = []
    for cy in row_centers:
        if not row_groups or cy - row_groups[-1] > tol:
            row_groups.append(cy)

    col_centers = sorted(set(x + w // 2 for x, y, w, h in cells))
    col_groups = []
    for cx in col_centers:
        if not col_groups or cx - col_groups[-1] > tol:
            col_groups.append(cx)

    grid = [[None] * len(col_groups) for _ in range(len(row_groups))]

    def nearest(lst, val):
        return min(range(len(lst)), key=lambda i: abs(lst[i] - val))

    for (x, y, w, h) in cells:
        r = nearest(row_groups, y + h // 2)
        c = nearest(col_groups, x + w // 2)
        grid[r][c] = (x, y, w, h)
    return grid


def ocr_cell(crop_img, x, y, w, h):
    cell = crop_img.crop((max(0, x-2), max(0, y-2),
                          min(crop_img.width, x+w+2), min(crop_img.height, y+h+2)))
    text = pytesseract.image_to_string(ImageOps.grayscale(cell), lang='kor+eng', config=TESS_CONFIG)
    return re.sub(r'\s+', ' ', text).strip()


def grid_to_markdown(grid, crop_img):
    if not grid or not grid[0]:
        return None
    rows_text = []
    for row in grid:
        cells_text = [ocr_cell(crop_img, *cell) if cell else "" for cell in row]
        rows_text.append(cells_text)
    if not rows_text:
        return None
    def row_str(r):
        return "| " + " | ".join(r) + " |"
    sep = ["---"] * len(rows_text[0])
    lines = [row_str(rows_text[0]), row_str(sep)] + [row_str(r) for r in rows_text[1:]]
    return "\n".join(lines)


def extract_table(img, element):
    crop = crop_element(img, element["bbox_px"])
    cells = detect_cells(crop)
    if not cells:
        return None
    grid = cells_to_grid(cells, crop.height)
    return grid_to_markdown(grid, crop)

## 2. 단일 페이지 테스트

In [ ]:
img = page_to_image(doc, TEST_PAGE, dpi=300)

# 탐색(100 DPI)과 추출(300 DPI) 해상도 비율로 bbox 스케일 조정
SCAN_DPI, EXTRACT_DPI = 100, 300
scale = EXTRACT_DPI / SCAN_DPI

table_elements = []
for i, bbox in enumerate(TEST_REGIONS):
    x1, y1, x2, y2 = [round(v * scale) for v in bbox]
    table_elements.append({
        "id": f"p{TEST_PAGE+1}_t{i+1}",
        "type": "table",
        "bbox_px": [x1, y1, x2, y2]
    })

print(f"[{TEST_PAGE+1}페이지] 표 영역: {len(table_elements)}개")

In [ ]:
for elem in table_elements:
    markdown = extract_table(img, elem)
    if markdown:
        elem["text"] = markdown
        print(f"[{elem['id']}] 추출 성공")
        print(markdown[:400])
    else:
        elem["text"] = None
        print(f"[{elem['id']}] 선 감지 실패 (테두리 없는 표일 가능성)")
    print()

## 3. 시각화 (셀 감지 확인)

In [ ]:
for elem in table_elements:
    crop = crop_element(img, elem["bbox_px"])
    cells = detect_cells(crop)

    vis = np.array(crop).copy()
    for (x, y, cw, ch) in cells:
        cv2.rectangle(vis, (x, y), (x+cw, y+ch), (0, 200, 0), 2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(crop)
    axes[0].set_title(f"{elem['id']} — 원본")
    axes[0].axis("off")
    axes[1].imshow(vis)
    axes[1].set_title(f"감지된 셀: {len(cells)}개")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{elem['id']}_cells.png"), dpi=120, bbox_inches="tight")
    plt.show()

## 4. 배치 처리 (기존 batch_formula.json 업데이트)

In [ ]:
TABLE_LABELS = {"table"}

batch_formula_path = os.path.join(FORMULA_DIR, "batch_formula.json")
with open(batch_formula_path, encoding="utf-8") as f:
    all_pages = json.load(f)

for page_data in all_pages:
    page_num = page_data["page_id"] - 1
    table_elems = [e for e in page_data["elements"] if e["type"] in TABLE_LABELS]

    if not table_elems:
        print(f"[{page_num+1}페이지] 표 없음 — 스킵")
        continue

    print(f"[{page_num+1}페이지] 표 {len(table_elems)}개 처리 중...", end=" ")
    img = page_to_image(doc, page_num)

    for elem in table_elems:
        try:
            elem["text"] = extract_table(img, elem)
        except Exception as e:
            elem["text"] = f"ERROR: {e}"

    print("완료")

batch_out = os.path.join(OUTPUT_DIR, "batch_table.json")
with open(batch_out, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

print(f"\n배치 저장 완료: {batch_out}")